### 1. Imports

In [1]:
import sys
import pandas as pd

from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid

sys.path.insert(0, r'D:\03_Python\sakura_analytics\src')

from model_data import (
    load_features,
    feature_columns,
    cv_folds,
    evaluate,
    TARGET,
)

### 2. Create the four datasets

In [2]:
df_feb_large = load_features('feb1', source='csv')
df_mar_large = load_features('mar1', source='csv')

feb_stations_2025 = df_feb_large.loc[
    df_feb_large['bloom_year'] == 2025,
    'station_id',
].unique()

mar_stations_2025 = df_mar_large.loc[
    df_mar_large['bloom_year'] == 2025,
    'station_id',
].unique()

df_feb_small = df_feb_large[
    df_feb_large['station_id'].isin(feb_stations_2025)
].copy()

df_mar_small = df_mar_large[
    df_mar_large['station_id'].isin(mar_stations_2025)
].copy()

# Execution order
datasets = {
    'Mar-large': df_mar_large,
    'Mar-small': df_mar_small,
    'Feb-large': df_feb_large,
    'Feb-small': df_feb_small,
}

dataset_summary = []

for name, data in datasets.items():
    dataset_summary.append({
        'dataset': name,
        'rows': len(data),
        'stations': data['station_id'].nunique(),
        'development_rows': (data['split'] == 'development').sum(),
        'test_rows': (data['split'] == 'test').sum(),
    })

display(pd.DataFrame(dataset_summary))

Loaded model_features_feb1 from data/model/ CSV: 4,365 rows
Loaded model_features_mar1 from data/model/ CSV: 4,365 rows


,dataset,rows,stations,development_rows,test_rows
0,Mar-large,4365,82,3837,528
1,Mar-small,2972,48,2444,528
2,Feb-large,4365,82,3837,528
3,Feb-small,2972,48,2444,528


### 3. Set the shared parameter grid

In [4]:
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 10, 30, 50],
    'max_features': ['sqrt', 0.7, 1.0],
}

candidates = list(ParameterGrid(param_grid))

print('Candidates per dataset:', len(candidates))
print('Total model fits:', len(candidates) * 5 * len(datasets))

Candidates per dataset: 96
Total model fits: 1920


### 4. Run grid search and display results for each dataset

In [6]:
all_results = {}

for dataset_name, df in datasets.items():
    display(Markdown(f'## {dataset_name}'))

    X = feature_columns(df)
    folds = list(cv_folds(df))

    for fold, train, valid in folds:
        assert not train.empty and not valid.empty
        assert train['bloom_year'].max() < valid['bloom_year'].min()
        assert valid['bloom_year'].max() <= 2014

    fold_rows = []

    for candidate_id, params in enumerate(candidates):
        candidate_mae = []

        for fold, train, valid in folds:
            model = RandomForestRegressor(
                **params,
                random_state=42,
                n_jobs=-1,
            )

            model.fit(train[X], train[TARGET])

            train_scores = evaluate(
                train[TARGET],
                model.predict(train[X]),
            )

            valid_scores = evaluate(
                valid[TARGET],
                model.predict(valid[X]),
            )

            fold_rows.append({
                'candidate_id': candidate_id,
                'fold': fold,
                'train_MAE': train_scores['MAE'],
                'valid_MAE': valid_scores['MAE'],
                'valid_RMSE': valid_scores['RMSE'],
                'valid_R2': valid_scores['R2'],
                'valid_hit_rate_3d': valid_scores['hit_rate_3d'],
            })

            candidate_mae.append(valid_scores['MAE'])

        # Show progress every six candidates.
        if (candidate_id + 1) % 6 == 0:
            mean_mae = sum(candidate_mae) / len(candidate_mae)

            print(
                f'{dataset_name} | '
                f'{candidate_id + 1}/{len(candidates)} | '
                f'Current candidate MAE: {mean_mae:.3f}',
                flush=True,
            )

    fold_results = pd.DataFrame(fold_rows)

    mean_scores = (
        fold_results
        .groupby('candidate_id')
        .agg(
            train_MAE=('train_MAE', 'mean'),
            valid_MAE=('valid_MAE', 'mean'),
            valid_MAE_std=('valid_MAE', 'std'),
            valid_RMSE=('valid_RMSE', 'mean'),
            valid_R2=('valid_R2', 'mean'),
            valid_hit_rate_3d=('valid_hit_rate_3d', 'mean'),
        )
    )

    parameter_table = pd.DataFrame(candidates)
    parameter_table.index.name = 'candidate_id'

    gs_results = (
        parameter_table
        .join(mean_scores)
        .sort_values('valid_MAE', kind='stable')
    )

    best_id = int(gs_results.index[0])
    best_params = candidates[best_id].copy()

    best_fold_results = fold_results[
        fold_results['candidate_id'] == best_id
    ].copy()

    all_results[dataset_name] = {
        'features': X.copy(),
        'best_id': best_id,
        'best_params': best_params,
        'gs_results': gs_results,
        'fold_results': fold_results,
        'best_fold_results': best_fold_results,
    }

    print('Best parameters:', best_params)

    display(Markdown('### Top 10 candidates'))
    display(gs_results.head(10))

    display(Markdown('### Fold results for the best candidate'))
    display(best_fold_results)

## Mar-large

Mar-large | 6/96 | Current candidate MAE: 4.055
Mar-large | 12/96 | Current candidate MAE: 4.009
Mar-large | 18/96 | Current candidate MAE: 4.026
Mar-large | 24/96 | Current candidate MAE: 3.966
Mar-large | 30/96 | Current candidate MAE: 3.958
Mar-large | 36/96 | Current candidate MAE: 3.902
Mar-large | 42/96 | Current candidate MAE: 3.964
Mar-large | 48/96 | Current candidate MAE: 3.885
Mar-large | 54/96 | Current candidate MAE: 3.959
Mar-large | 60/96 | Current candidate MAE: 3.902
Mar-large | 66/96 | Current candidate MAE: 3.979
Mar-large | 72/96 | Current candidate MAE: 3.885
Mar-large | 78/96 | Current candidate MAE: 3.958
Mar-large | 84/96 | Current candidate MAE: 3.900
Mar-large | 90/96 | Current candidate MAE: 3.987
Mar-large | 96/96 | Current candidate MAE: 3.885
Best parameters: {'max_depth': 10, 'max_features': 1.0, 'min_samples_leaf': 30, 'n_estimators': 200}


### Top 10 candidates

,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
44,10.0,1.0,30,200,2.958551,3.878448,1.799709,4.645620,0.798799,0.469847
92,NaN,1.0,30,200,2.946756,3.878455,1.794543,4.644827,0.798843,0.469847
68,15.0,1.0,30,200,2.946760,3.878462,1.794539,4.644832,0.798843,0.469847
93,NaN,1.0,30,500,2.946926,3.878847,1.804047,4.646730,0.798660,0.472354
69,15.0,1.0,30,500,2.946931,3.878874,1.804047,4.646753,0.798659,0.472354
45,10.0,1.0,30,500,2.959103,3.879099,1.808875,4.648218,0.798563,0.470313
47,10.0,1.0,50,500,3.118179,3.885155,1.783763,4.669235,0.798057,0.481510
71,15.0,1.0,50,500,3.117686,3.885300,1.783607,4.669306,0.798048,0.481510
95,NaN,1.0,50,500,3.117686,3.885300,1.783607,4.669306,0.798048,0.481510


### Fold results for the best candidate

...

Feb-small | 6/96 | Current candidate MAE: 3.921
Feb-small | 12/96 | Current candidate MAE: 3.832
Feb-small | 18/96 | Current candidate MAE: 3.882
Feb-small | 24/96 | Current candidate MAE: 3.857
Feb-small | 30/96 | Current candidate MAE: 3.933
Feb-small | 36/96 | Current candidate MAE: 3.950
Feb-small | 42/96 | Current candidate MAE: 4.123
Feb-small | 48/96 | Current candidate MAE: 3.899
Feb-small | 54/96 | Current candidate MAE: 3.939
Feb-small | 60/96 | Current candidate MAE: 3.978
Feb-small | 66/96 | Current candidate MAE: 4.236
Feb-small | 72/96 | Current candidate MAE: 3.899
Feb-small | 78/96 | Current candidate MAE: 3.939
Feb-small | 84/96 | Current candidate MAE: 3.985
Feb-small | 90/96 | Current candidate MAE: 4.245
Feb-small | 96/96 | Current candidate MAE: 3.899
Best parameters: {'max_depth': 5, 'max_features': 0.7, 'min_samples_leaf': 30, 'n_estimators': 500}


### Top 10 candidates

,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
13,5.0,0.7,30,500,3.589641,3.830704,1.707130,4.651702,0.811132,0.483333
21,5.0,1.0,30,500,3.580374,3.830736,1.700003,4.630306,0.813110,0.479167
11,5.0,0.7,10,500,3.477056,3.831906,1.686526,4.649140,0.811951,0.481250
20,5.0,1.0,30,200,3.584127,3.835295,1.696501,4.636914,0.812741,0.477083
12,5.0,0.7,30,200,3.592316,3.839127,1.708461,4.655707,0.811015,0.475000
18,5.0,1.0,10,200,3.457456,3.841002,1.675199,4.641779,0.812889,0.477083
19,5.0,1.0,10,500,3.456804,3.841260,1.674793,4.640901,0.812987,0.475000
10,5.0,0.7,10,200,3.477294,3.844062,1.685771,4.659895,0.811357,0.481250
15,5.0,0.7,50,500,3.641309,3.844753,1.691882,4.675745,0.809432,0.479167


### Fold results for the best candidate

,candidate_id,fold,train_MAE,valid_MAE,valid_RMSE,valid_R2,valid_hit_rate_3d
65,13,1,3.643731,2.724374,3.710342,0.893010,0.677083
66,13,2,3.588444,3.793844,4.718808,0.743723,0.468750
67,13,3,3.593063,3.750620,4.836587,0.818896,0.489583
68,13,4,3.593450,2.242654,2.707690,0.924642,0.677083
69,13,5,3.529516,6.642026,7.285083,0.675387,0.104167


In [7]:
comparison_rows = []

for dataset_name, result in all_results.items():
    best = result['gs_results'].iloc[0]

    comparison_rows.append({
        'dataset': dataset_name,
        **result['best_params'],
        'train_MAE': best['train_MAE'],
        'valid_MAE': best['valid_MAE'],
        'valid_MAE_std': best['valid_MAE_std'],
        'valid_RMSE': best['valid_RMSE'],
        'valid_R2': best['valid_R2'],
        'valid_hit_rate_3d': best['valid_hit_rate_3d'],
    })

comparison = pd.DataFrame(comparison_rows).set_index('dataset')

display(comparison.round(3))


,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
dataset,,,,,,,,,,
Mar-large,10,1.0,30,200,2.959,3.878,1.800,4.646,0.799,0.470
Mar-small,10,sqrt,1,200,2.010,3.880,1.861,4.671,0.796,0.481
Feb-large,10,0.7,50,500,3.584,3.876,1.786,4.696,0.803,0.487
Feb-small,5,0.7,30,500,3.590,3.831,1.707,4.652,0.811,0.483


In [8]:
datasets_30y = {
    'Feb-large-30y': df_feb_large,
    'Mar-large-30y': df_mar_large,
}

# Use the same grid as the original four models.
candidates_30y = list(ParameterGrid(param_grid))

for dataset_name, df in datasets_30y.items():
    display(Markdown(f'## {dataset_name}'))

    X = feature_columns(df)
    folds_30y = []
    period_rows = []

    for fold, train, valid in cv_folds(df):
        last_train_year = int(train['bloom_year'].max())
        first_train_year = last_train_year - 29

        train_30y = train[
            train['bloom_year'].between(
                first_train_year,
                last_train_year,
            )
        ].copy()

        assert not train_30y.empty and not valid.empty
        assert train_30y['bloom_year'].max() < valid['bloom_year'].min()
        assert valid['bloom_year'].max() <= 2014

        folds_30y.append((fold, train_30y, valid))

        period_rows.append({
            'fold': fold,
            'train_start': first_train_year,
            'train_end': last_train_year,
            'valid_start': valid['bloom_year'].min(),
            'valid_end': valid['bloom_year'].max(),
            'train_rows': len(train_30y),
            'valid_rows': len(valid),
        })

    display(pd.DataFrame(period_rows))

    fold_rows = []

    for candidate_id, params in enumerate(candidates_30y):
        candidate_mae = []

        for fold, train, valid in folds_30y:
            model = RandomForestRegressor(
                **params,
                random_state=42,
                n_jobs=-1,
            )

            model.fit(train[X], train[TARGET])

            train_scores = evaluate(
                train[TARGET],
                model.predict(train[X]),
            )

            valid_scores = evaluate(
                valid[TARGET],
                model.predict(valid[X]),
            )

            fold_rows.append({
                'candidate_id': candidate_id,
                'fold': fold,
                'train_MAE': train_scores['MAE'],
                'valid_MAE': valid_scores['MAE'],
                'valid_RMSE': valid_scores['RMSE'],
                'valid_R2': valid_scores['R2'],
                'valid_hit_rate_3d': valid_scores['hit_rate_3d'],
            })

            candidate_mae.append(valid_scores['MAE'])

        if (
            (candidate_id + 1) % 6 == 0
            or candidate_id + 1 == len(candidates_30y)
        ):
            mean_mae = sum(candidate_mae) / len(candidate_mae)

            print(
                f'{dataset_name} | '
                f'{candidate_id + 1}/{len(candidates_30y)} | '
                f'Current candidate MAE: {mean_mae:.3f}',
                flush=True,
            )

    fold_results = pd.DataFrame(fold_rows)

    mean_scores = (
        fold_results
        .groupby('candidate_id')
        .agg(
            train_MAE=('train_MAE', 'mean'),
            valid_MAE=('valid_MAE', 'mean'),
            valid_MAE_std=('valid_MAE', 'std'),
            valid_RMSE=('valid_RMSE', 'mean'),
            valid_R2=('valid_R2', 'mean'),
            valid_hit_rate_3d=('valid_hit_rate_3d', 'mean'),
        )
    )

    parameter_table = pd.DataFrame(candidates_30y)
    parameter_table.index.name = 'candidate_id'

    gs_results = (
        parameter_table
        .join(mean_scores)
        .sort_values('valid_MAE', kind='stable')
    )

    best_id = int(gs_results.index[0])
    best_params = candidates_30y[best_id].copy()

    best_fold_results = fold_results[
        fold_results['candidate_id'] == best_id
    ].copy()

    # Add results without removing the original four models.
    all_results[dataset_name] = {
        'features': X.copy(),
        'best_id': best_id,
        'best_params': best_params,
        'gs_results': gs_results,
        'fold_results': fold_results,
        'best_fold_results': best_fold_results,
        'training_periods': pd.DataFrame(period_rows),
    }

    print('Best parameters:', best_params)

    display(Markdown('### Top 10 candidates'))
    display(gs_results.head(10))

    display(Markdown('### Fold results for the best candidate'))
    display(best_fold_results)

## Feb-large-30y

,fold,train_start,train_end,valid_start,valid_end,train_rows,valid_rows
0,1,1975,2004,2005,2006,2391,148
1,2,1977,2006,2007,2008,2378,127
2,3,1979,2008,2009,2010,2344,106
3,4,1981,2010,2011,2012,2289,98
4,5,1983,2012,2013,2014,2225,96


Feb-large-30y | 6/96 | Current candidate MAE: 3.961
Feb-large-30y | 12/96 | Current candidate MAE: 3.858
Feb-large-30y | 18/96 | Current candidate MAE: 3.936
Feb-large-30y | 24/96 | Current candidate MAE: 3.899
Feb-large-30y | 30/96 | Current candidate MAE: 3.939
Feb-large-30y | 36/96 | Current candidate MAE: 3.972
Feb-large-30y | 42/96 | Current candidate MAE: 4.116
Feb-large-30y | 48/96 | Current candidate MAE: 3.905
Feb-large-30y | 54/96 | Current candidate MAE: 3.941
Feb-large-30y | 60/96 | Current candidate MAE: 4.019
Feb-large-30y | 66/96 | Current candidate MAE: 4.225
Feb-large-30y | 72/96 | Current candidate MAE: 3.906
Feb-large-30y | 78/96 | Current candidate MAE: 3.941
Feb-large-30y | 84/96 | Current candidate MAE: 4.019
Feb-large-30y | 90/96 | Current candidate MAE: 4.248
Feb-large-30y | 96/96 | Current candidate MAE: 3.906
Best parameters: {'max_depth': 5, 'max_features': 0.7, 'min_samples_leaf': 30, 'n_estimators': 500}


### Top 10 candidates

,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
13,5.0,0.7,30,500,3.780959,3.838836,1.427245,4.730592,0.806479,0.475029
12,5.0,0.7,30,200,3.781017,3.839919,1.423745,4.726731,0.807004,0.473143
15,5.0,0.7,50,500,3.864933,3.851163,1.438732,4.759567,0.803938,0.479684
63,15.0,0.7,50,500,3.695883,3.857144,1.542146,4.724787,0.804497,0.469206
87,NaN,0.7,50,500,3.695883,3.857144,1.542146,4.724787,0.804497,0.469206
39,10.0,0.7,50,500,3.697240,3.857158,1.541190,4.724559,0.804566,0.471247
11,5.0,0.7,10,500,3.679300,3.857512,1.366260,4.758156,0.804762,0.482569
36,10.0,0.7,30,200,3.512473,3.861787,1.508968,4.692296,0.807307,0.467878
10,5.0,0.7,10,200,3.681998,3.863071,1.365990,4.760458,0.804659,0.479642


...

Mar-large-30y | 6/96 | Current candidate MAE: 4.141
Mar-large-30y | 12/96 | Current candidate MAE: 4.149
Mar-large-30y | 18/96 | Current candidate MAE: 4.177
Mar-large-30y | 24/96 | Current candidate MAE: 4.153
Mar-large-30y | 30/96 | Current candidate MAE: 4.120
Mar-large-30y | 36/96 | Current candidate MAE: 4.133
Mar-large-30y | 42/96 | Current candidate MAE: 4.160
Mar-large-30y | 48/96 | Current candidate MAE: 4.124
Mar-large-30y | 54/96 | Current candidate MAE: 4.119
Mar-large-30y | 60/96 | Current candidate MAE: 4.143
Mar-large-30y | 66/96 | Current candidate MAE: 4.182
Mar-large-30y | 72/96 | Current candidate MAE: 4.124
Mar-large-30y | 78/96 | Current candidate MAE: 4.119
Mar-large-30y | 84/96 | Current candidate MAE: 4.144
Mar-large-30y | 90/96 | Current candidate MAE: 4.186
Mar-large-30y | 96/96 | Current candidate MAE: 4.124
Best parameters: {'max_depth': 15, 'max_features': 1.0, 'min_samples_leaf': 30, 'n_estimators': 200}


### Top 10 candidates

,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
candidate_id,,,,,,,,,,
68,15.0,1.0,30,200,2.800782,4.090135,1.541486,4.861139,0.780882,0.425478
92,NaN,1.0,30,200,2.800782,4.090135,1.541486,4.861139,0.780882,0.425478
44,10.0,1.0,30,200,2.801082,4.090355,1.541446,4.861229,0.780873,0.425478
93,NaN,1.0,30,500,2.799348,4.097624,1.540359,4.871914,0.779763,0.423819
69,15.0,1.0,30,500,2.799348,4.097624,1.540359,4.871914,0.779763,0.423819
45,10.0,1.0,30,500,2.799601,4.097690,1.540228,4.871949,0.779761,0.423819
74,NaN,sqrt,10,200,2.483241,4.097700,1.571329,4.900926,0.776857,0.436221
51,15.0,sqrt,10,500,2.484230,4.097796,1.566040,4.901307,0.776776,0.436248
75,NaN,sqrt,10,500,2.483199,4.098137,1.568780,4.903380,0.776517,0.434361


### Fold results for the best candidate

,candidate_id,fold,train_MAE,valid_MAE,valid_RMSE,valid_R2,valid_hit_rate_3d
340,68,1,2.837859,2.360617,3.154919,0.926467,0.716216
341,68,2,2.755719,4.840889,5.803561,0.609198,0.330709
342,68,3,2.836118,3.142901,3.951744,0.871687,0.537736
343,68,4,2.802509,3.788642,4.478910,0.789659,0.448980
344,68,5,2.771703,6.317623,6.916560,0.707398,0.093750


In [9]:
comparison_rows = []

for dataset_name, result in all_results.items():
    best = result['gs_results'].iloc[0]

    comparison_rows.append({
        'dataset': dataset_name,
        **result['best_params'],
        'train_MAE': best['train_MAE'],
        'valid_MAE': best['valid_MAE'],
        'valid_MAE_std': best['valid_MAE_std'],
        'valid_RMSE': best['valid_RMSE'],
        'valid_R2': best['valid_R2'],
        'valid_hit_rate_3d': best['valid_hit_rate_3d'],
    })

comparison = pd.DataFrame(comparison_rows).set_index('dataset')

display(comparison.round(3))

,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,valid_MAE,valid_MAE_std,valid_RMSE,valid_R2,valid_hit_rate_3d
dataset,,,,,,,,,,
Mar-large,10,1.0,30,200,2.959,3.878,1.800,4.646,0.799,0.470
Mar-small,10,sqrt,1,200,2.010,3.880,1.861,4.671,0.796,0.481
Feb-large,10,0.7,50,500,3.584,3.876,1.786,4.696,0.803,0.487
Feb-small,5,0.7,30,500,3.590,3.831,1.707,4.652,0.811,0.483
Feb-large-30y,5,0.7,30,500,3.781,3.839,1.427,4.731,0.806,0.475
Mar-large-30y,15,1.0,30,200,2.801,4.090,1.541,4.861,0.781,0.425


In [ ]:
df_result = comparison.reset_index()
df_result.to_csv('../../results/RF_results.csv', index=False)